<td>
<a href="https://colab.research.google.com/github/raoulg/MADS-DAV/blob/main/notebooks/lesson6/06.1-dimensionality_reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
</td>

# 6.1 Dimensionality reduction, and what it does to you

Every dataset so far had few enough columns to plot. This one does not: a handwritten digit
is 784 numbers, and there is no scatter plot of 784 dimensions. Two methods squash that into
a picture you can look at.

Both work. Both also have a decision inside them that they will not make for you, and both
produce a confident-looking picture either way:

- **PCA** finds the directions along which the data varies most — measured in whatever units
  you handed it. Give it grams and it will find grams.
- **t-SNE** arranges points so that neighbours stay neighbours. How many neighbours is a
  parameter, and the shape of the answer depends on it.

The sections named "where it goes wrong" are staged: the failure is deliberate and the
correct version is right next to it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from goad_toolkit.visualizer import PlotSettings, ProjectionPlot, ScreePlot
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from wa_analyzer.data import load_showcase

# SVD and PCA
Singular Value Decomposition, or SVD for short, is a mathematical technique that decomposes a matrix into three simpler matrices. Imagine you have a large, complex puzzle; SVD helps you break it down into smaller, easier-to-understand pieces.

Single Value Decomposition (SVD) is defined like this:

$$X = U \Sigma V^T$$

U (left singular vectors): This is a matrix that contains information about the patterns and relationships between the rows of the original matrix.

Σ (singular values): This is a diagonal matrix with non-negative numbers, which can be seen as the "strength" or "importance" of each pattern found by U and V. The singular values are sorted from largest to smallest, showing the ranking of the importance of each pattern.

V^T (right singular vectors, transpose of V): This is a matrix that contains information about the patterns and relationships between the columns of the original matrix.

What are some applications where you could encounter SVD?

1. Data compression: SVD can help you reduce the size of the data without losing much information. By keeping only the largest singular values (and associated vectors), you can get a compact version of your original matrix.
2. Noise reduction: SVD can help remove noise from data. In this context, noise is represented by the components with smaller singular values that you can discard, leaving you with the more significant, stronger signals in your data.
3. Latent semantic analysis: In natural language processing, SVD is used to understand the relationships between documents and terms in text data, helping to find the hidden (or "latent") concepts.
4. Principal Component Analysis (PCA): In machine learning, SVD is used as part of PCA, which is a method that reduces the dimensionality of data while retaining most of the variation in the dataset.

SVD is like a Swiss Army knife for matrices

## Explanation of some terminology

$U$ provides an orthonormal basis for the column space of $X$, and V provides an orthonormal basis for the row space of $X$.

Some explanation of what an orthonormal basis means in simple terms:

- *Orthogonal*: Each line is at a 90-degree angle to the others, like the corners of a room. One line goes from wall to wall, another from floor to ceiling, and another from one corner of the room to the opposite corner if it's a 3D space. They don't lean toward each other at all. This is how we typically think of axis lines in a graph. Note that they don't NEED to be at a 90 degree angle, but it doesnt make much sense to have them at any other angle.

- *Normalized*: Each line has been stretched or shrunk to exactly the same length. Let's say we've decided that the length is the length of a meter stick. So, if you were to walk from the center of the room to the wall following one of these lines, you'd always walk exactly one meter, no matter which line you chose.

- *Basis*: Using these lines, you can reach any point in the room by walking along them one at a time. In a grid with 3D axis, the basis would be the vectors (0,0,1), (0,1,0) and (1,0,0). If you want to reach any point in the room, you can reach it by combining these three vectors.

The key characteristics of a basis are:
- *Coverage*: A basis must be able to reach or "span" every point in the space. Just like with our north and east directions on the floor, by going some amount north and some amount east, you can reach any point on the floor. If you were in a three-dimensional room, you'd also need an up-and-down direction to reach every point.

- *Independence*: The directions in a basis must be independent of each other, meaning you can't create one direction just by using a combination of the others. On our floor, you can't get a north direction just by walking east or vice versa; they're completely separate.

# Examples

This is an illustration of how minimizing the distance of the red dotted line is the same as maximizing the green dotted line.

In [ ]:

# Define the line L (y = mx + c form)
m, c = 1, 0  # slope and y-intercept
x = np.linspace(-5, 5, 400)
y = m * x + c

# Define the point P
P = np.array([2, 3])

# Calculate the projection P' onto the line L
# Line's normal vector
normal = np.array([-m, 1])
# Projection of P onto the line (using dot product)
P_proj = P - np.dot(P, normal) / np.dot(normal, normal) * normal

# Plotting
plt.figure(figsize=(8, 8))
plt.plot(x, y, label="Line L (y = x)", color="blue")  # Line
plt.scatter(*P, color="red", label="Point P")  # Point P
plt.scatter(*P_proj, color="green", label="Projected Point P'")  # Projected Point P'
plt.plot(
    [P[0], P_proj[0]],
    [P[1], P_proj[1]],
    color="red",
    linestyle="--",
    label="Minimized Distance",
)  # Minimized Distance
plt.plot(
    [0, P_proj[0]],
    [0, P_proj[1]],
    color="green",
    linestyle="--",
    label="Maximized Distance",
)  # Maximized Distance

# Annotations and decorations
plt.scatter(0, 0, color="black", label="Origin O")  # Origin
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.grid(True)
plt.legend()
plt.title(
    "Minimizing Distance from Point to Line Projection\n and Maximizing Distance from Origin to Projected Point"
)

# Show plot
plt.show()

Lets make some synthetic data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def synthetic_data(seed: int = 42, m: int = 60) -> np.ndarray:
    np.random.seed(seed)
    # two weigths
    w1, w2 = 0.1, 0.3
    # some noise
    noise = 0.1

    # m random angles
    angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
    X = np.empty((m, 3))  # noqa: N806
    X[:, 0] = np.cos(angles) + np.sin(angles) / 2 + noise * np.random.randn(m) / 2
    X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
    X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)
    return X


spiral = synthetic_data(seed=4, m=100)

Run a SVD on it with numpy

In [ ]:
spiral_centered = spiral - spiral.mean(axis=0)
U, s, Vt = np.linalg.svd(spiral_centered)
U.shape, s.shape, Vt.shape

And lets visualize both the data, and the eigenvectors

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

# Scatter plot of the original data
ax.scatter(spiral_centered[:, 0], spiral_centered[:, 1], spiral_centered[:, 2])

# The principal components are the rows of Vt. We scale them by the square root of the eigenvalues (s**2).
for i in range(Vt.shape[0]):
    # Start the line in the middle of the data
    start_point = np.zeros(3)
    # The end of the line is the direction of the principal component
    end_point = Vt[i, :]
    # Plot the principal components as lines
    ax.quiver(
        start_point[0],
        start_point[1],
        start_point[2],
        end_point[0],
        end_point[1],
        end_point[2],
        color=["r", "g", "b"][i],
        arrow_length_ratio=0.05,
        linewidths=3,
    )

# Set labels for axes
ax.set_xlabel("X axis")
ax.set_ylabel("Y axis")
ax.set_zlabel("Z axis")  # ty: ignore[unresolved-attribute] -- Axes3D, not the base Axes the stub sees

# Show the plot
plt.show()

As you can see, the data (blue points) are spread out over a diagonal surface. With SVD we are able to find three vectors that are orthogonal to each other, and that span the space of the data. They also make intuitively the most sense; the red vector covers more of the data than the X or Y axis does.

We can take the first two eigenvectors and project the data onto them

In [ ]:
W2 = Vt.T[:, :2]
X2D_svd = spiral_centered.dot(W2)

settings = PlotSettings(
    figsize=(6, 5),
    title="The spiral, projected onto its first two singular vectors",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=X2D_svd, color="#4c72b0")

As you can see, this is a nice way to "project" the data onto a lower dimension (in this case, from 3D to 2D) while retaining most of the information. This is the basis of PCA.
Now, PCA with two components should give the same result:

In [ ]:
pca = PCA(n_components=2)
X2D = pca.fit_transform(spiral)
np.allclose(X2D_svd, X2D) or np.allclose(X2D_svd, -X2D)

Even though the y-axis is sometimes flipped, the data is the same

We can look up the explained variance ratio

In [ ]:
pca = PCA().fit(spiral)
print("explained variance ratio per component:")
print(np.round(pca.explained_variance_ratio_, 4))
print("\nfrom the singular values, squared and normalised:")
print(np.round(np.square(s) / np.sum(np.square(s)), 4))

settings = PlotSettings(
    figsize=(6, 4),
    title=f"Two components carry {pca.explained_variance_ratio_[:2].sum():.1%} of the variance",
    xlabel="component",
    ylabel="share of variance",
)
fig, ax = ScreePlot(settings).plot(explained_variance_ratio=pca.explained_variance_ratio_)

# Where PCA goes wrong: the units you gave it

PCA maximises variance. Variance has units — squared ones — so the component it finds is
partly a fact about the data and partly a fact about what you measured things in. On the
spiral above that was invisible, because all three columns were the same made-up scale.

The penguins from lesson 2 are not. Look at the standard deviations before doing anything
else.

In [ ]:
penguins = load_showcase("penguins").dropna()
FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
print(penguins[FEATURES].describe().loc[["count", "mean", "std"]].round(1).to_string())

In [ ]:
measurements = penguins[FEATURES].to_numpy()
unscaled = PCA().fit(measurements)

print(f"explained variance ratio: {np.round(unscaled.explained_variance_ratio_, 4)}")
print("\nwhat the first component is made of:")
print(pd.Series(unscaled.components_[0], index=FEATURES).round(3).to_string())

**The first component explains 99.99% of the variance and it is body mass**, with a loading
of 1.000 against roughly zero for everything else. PCA did not fail — it did exactly what it
promises. Body mass has a standard deviation of 805, bill depth has 2.0, so essentially all
the variance in this matrix is grams, and the direction of greatest variance is the grams
axis.

A 99.99% number in a report looks like a triumph. It is the symptom.

The clearest way to see that this is about units and not about penguins: divide body mass by
a thousand and call it kilograms. Same birds, same measurements, same information.

In [ ]:
kilos = measurements.copy()
kilos[:, FEATURES.index("body_mass_g")] /= 1000
in_kilos = PCA().fit(kilos)

print(f"explained variance ratio: {np.round(in_kilos.explained_variance_ratio_, 4)}")
print("\nwhat the first component is made of now:")
print(pd.Series(in_kilos.components_[0], index=FEATURES).round(3).to_string())

The first component is now **flipper length**, at a loading of 0.96, and it explains 91.9%
instead of 99.99%. Nothing about the birds changed. A unit did.

So before PCA, put every column on the same footing: subtract the mean and divide by the
standard deviation, which is `StandardScaler`. After that a "unit" of every column is one
standard deviation of that column, and no column can win by being measured in something small.

In [ ]:
standardised = StandardScaler().fit_transform(measurements)
scaled = PCA().fit(standardised)

print(f"explained variance ratio: {np.round(scaled.explained_variance_ratio_, 4)}\n")
loadings = pd.DataFrame(scaled.components_[:2].T, index=pd.Index(FEATURES), columns=pd.Index(["PC1", "PC2"]))
print(loadings.round(2).to_string())

Now the components mean something you can say out loud.

**PC1 is size**: bill length, flipper length and body mass all load positively, so a penguin
high on PC1 is a big penguin. Bill *depth* loads negatively — which is 5.2's finding
arriving from a different direction, because the long-billed species is the one with shallow
bills.

**PC2 is bill shape**: length and depth together, flipper and mass at nothing. It is the
"how chunky is the beak" axis, and it is a real second thing about a penguin rather than a
rounding error on the first.

Two components, 88% of the variance, and a sentence per axis. Compare that with the
99.99% version, which had one axis and it was a scale.

In [ ]:
def separation(scores: np.ndarray) -> float:
    """Between-species variance over within-species variance, along one component."""
    frame = pd.DataFrame({"score": scores, "species": penguins.species.to_numpy()})
    return frame.groupby("species").score.mean().var() / frame.groupby("species").score.var().mean()


print(f"species separation along PC1, unscaled: "
      f"{separation(unscaled.transform(measurements)[:, 0]):.2f}")
print(f"species separation along PC1, scaled:   "
      f"{separation(scaled.transform(standardised)[:, 0]):.2f}")

settings = PlotSettings(
    figsize=(11, 4.5),  # ty: ignore[invalid-argument-type]
    title="The same 333 penguins, before and after scaling",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[
        f"raw units — PC1+PC2 = {unscaled.explained_variance_ratio_[:2].sum():.1%}",
        f"standardised — PC1+PC2 = {scaled.explained_variance_ratio_[:2].sum():.1%}",
    ],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=2)
for ax, model, values in [(axes[0], unscaled, measurements), (axes[1], scaled, standardised)]:
    host.plot_on_axes(ProjectionPlot(settings), ax,
                      coordinates=model.transform(values)[:, :2],
                      labels=penguins.species.to_numpy(), legend=(ax is axes[1]))

It is not only a matter of interpretation: the unscaled version is **worse at the job**. Along
PC1, the ratio of between-species to within-species variance goes from 3.09 to 9.50 once the
columns are standardised. On the left, Gentoo pulls away — they are the heavy ones — but
Adelie and Chinstrap stay thoroughly mixed, because the only thing separating them is bill
shape and bill shape is measured in millimetres. On the right all three come apart.

And look at what the left panel does not tell you. Its two axes are 99.99% and 0.01% of the
variance, so honestly drawn it would be a horizontal line; matplotlib stretched the second
axis to fill the panel, and a 0.01% component now looks like half the picture. **A projection
plot always fills its frame.** That is exactly why the scree plot is not optional.

> **The rule.** Standardise before PCA unless every column is already in the same unit *and*
> you mean for the bigger-varying ones to count more. Pixel intensities, all 0–255, are the
> usual exception — which is why the MNIST section below does not scale.

# The Swiss roll
Let's have a look at a synthetic dataset known as the swiss roll

In [ ]:
from sklearn.datasets import make_swiss_roll

roll, colour = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(roll[:, 0], roll[:, 1], roll[:, 2], c=colour, cmap=plt.cm.hot)  # ty: ignore[unresolved-attribute]
ax.view_init(10, -70)  # ty: ignore[unresolved-attribute] -- Axes3D, not the base Axes the stub sees
ax.set_title("Swiss Roll Dataset")

And run PCA on it

In [ ]:
pca = PCA(n_components=2)
roll_2d = pca.fit_transform(roll)

settings = PlotSettings(
    figsize=(6, 5),
    title=f"Swiss roll under PCA — {pca.explained_variance_ratio_.sum():.0%} of the variance",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=roll_2d, labels=colour, palette="hot",
                                        legend=False)

Now, lets try t-SNE to visualize

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
roll_tsne = tsne.fit_transform(roll)

settings = PlotSettings(
    figsize=(6, 5),
    title="Swiss roll under t-SNE",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=roll_tsne, labels=colour, palette="hot",
                                        legend=False)

It's pretty different! But the real value of t-SNE is that it can be used to visualize high-dimensional data, where the manifold is very complex and non-linear.

First, download the data

In [ ]:
from sklearn.datasets import fetch_openml
from pathlib import Path

cache = Path.home() / ".cache/mads-dav"
if not cache.exists():
    cache.mkdir(parents=True)


mnist = fetch_openml("mnist_784", version=1, as_frame=False, data_home=str(cache))
mnist.target = mnist.target.astype(np.uint8)

In [ ]:
digits = mnist["data"]
digit_labels = mnist["target"]
print(f"{digits.shape[0]:,} digits, each one {digits.shape[1]} pixels")

Seventy thousand digits is more than t-SNE wants: it is roughly quadratic in the number of
points, and this notebook fits it six times. Five thousand, drawn at random, is enough to see
everything below and turns a coffee break into a few seconds.

That is a decision too. Write it down when you make it — a picture of a sample is a picture of
a sample.

In [ ]:
rng = np.random.default_rng(42)
sample = rng.choice(len(digits), size=5000, replace=False)
subset, subset_labels = digits[sample], digit_labels[sample]
print(f"{subset.shape[0]:,} digits, {subset.shape[1]} pixels each")

Every digit is a 28x28 image flattened into a row of 784 numbers, one per pixel. Here is the
first one, folded back into a square:

In [ ]:
img = digits[0]
plt.imshow(img.reshape(28, 28), cmap="gray")
plt.show()

This is clearly an handwritten digit, and we represent it as a matrix in $\mathbb{R}^{28 \times 28}$, or as a vector in $\mathbb{R}^{784}$. Now, lets try to see what happens if we visualise the data in 2D with PCA.

In [ ]:
pca = PCA().fit(subset)
cumulative = np.cumsum(pca.explained_variance_ratio_)

print(f"the first two components carry {cumulative[1]:.1%} of the variance")
for target in (0.5, 0.8, 0.95):
    print(f"  components needed for {target:.0%}: {int(np.argmax(cumulative >= target)) + 1}")

settings = PlotSettings(
    figsize=(7, 4),
    title="784 pixels, and how the variance is spread over them",
    xlabel="component",
    ylabel="share of variance",
)
fig, ax = ScreePlot(settings).plot(explained_variance_ratio=pca.explained_variance_ratio_,
                                   n_components=20)

In [ ]:
settings = PlotSettings(
    figsize=(7, 6),
    title=f"MNIST under PCA — two components, {cumulative[1]:.1%} of the variance",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=pca.transform(subset)[:, :2],
                                        labels=subset_labels, palette="tab10", s=8)

The digits overlap into a single smear, and the scree plot says why: **two components carry
17.0% of the variance**, and you need 148 of them for 95%. The picture is not wrong; it is a
picture of 17% of this dataset, and that number belongs in the title — which is what
`ScreePlot` and the title above it are for.

This is the honest version of "PCA does something but it is not very useful in two
dimensions". It is not useless; it is showing you the 17% that fits.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, init="pca")
mnist_tsne = tsne.fit_transform(subset)

settings = PlotSettings(
    figsize=(7, 6),
    title="MNIST under t-SNE, default perplexity of 30",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=mnist_tsne, labels=subset_labels,
                                        palette="tab10", s=8)

Ten clumps, one per digit, and the pairs that touch are the pairs that look alike. This is
t-SNE doing real work that PCA could not: the manifold is curved, and PCA can only cut it
with a flat plane.

Now the parameter.

# Where t-SNE goes wrong: perplexity is not a rendering option

Perplexity is roughly "how many neighbours each point should try to stay near". It has a
default of 30, and defaults are where decisions go to hide. Same 5,000 digits, four values.

In [ ]:
PERPLEXITIES = (5, 30, 100, 200)
embeddings = {
    perplexity: TSNE(n_components=2, perplexity=perplexity, random_state=42,
                     init="pca").fit_transform(subset)
    for perplexity in PERPLEXITIES
}

settings = PlotSettings(
    figsize=(11, 9),
    title="One dataset, four perplexities",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[f"perplexity {perplexity}" for perplexity in PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(PERPLEXITIES))
for ax, perplexity in zip(axes, PERPLEXITIES):
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embeddings[perplexity],
                      labels=subset_labels, palette="tab10", legend=False, s=8)

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score

pairs = np.triu_indices(10, k=1)
gaps = {}
for perplexity, embedding in embeddings.items():
    frame = pd.DataFrame(embedding, columns=pd.Index(["x", "y"])).assign(digit=subset_labels)
    centroids = frame.groupby("digit")[["x", "y"]].mean().to_numpy()
    distances = np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=-1)
    gaps[perplexity] = distances[pairs] / distances[pairs].max()
    print(f"perplexity {perplexity:>3d}: silhouette by true digit "
          f"{silhouette_score(embedding, subset_labels):.2f}")

print("\nspearman between the pairwise gaps each perplexity draws:")
table = pd.DataFrame(
    [[spearmanr(gaps[a], gaps[b]).statistic for b in PERPLEXITIES] for a in PERPLEXITIES],
    index=pd.Index(PERPLEXITIES), columns=pd.Index(PERPLEXITIES),
)
print(table.round(2).to_string())

names = [f"{a}-{b}" for a in range(10) for b in range(a + 1, 10)]
print("\nwhich digits the picture puts closest together, and furthest apart:")
for perplexity in (PERPLEXITIES[0], PERPLEXITIES[-1]):
    order = np.argsort(gaps[perplexity])
    print(f"  perplexity {perplexity:>3d}: closest {[names[i] for i in order[:3]]}, "
          f"furthest {[names[i] for i in order[-3:]]}")

Two different answers, and you need both.

**Membership is stable.** The silhouette scored against the true digit stays between 0.20 and
0.28 in all four. Whichever perplexity you pick, the ten groups come out as ten groups. When
you say "there are clusters here", t-SNE is backing you up.

**Geometry is not.** The gaps between those clusters, ranked, agree between perplexity 5 and
100 at a spearman of only 0.41. At perplexity 5 the two digits furthest apart are 0 and 1;
at perplexity 200 they are 0 and 7. So *how far apart* two clumps sit is a fact about the
parameter, not about handwriting.

The one thing that survives is 4-9 as the closest pair at both extremes, which is the
confusion any person would predict. A relationship that holds across settings is worth
reporting. The arrangement of a single picture is not.

> **What you may say about a t-SNE plot:** which points ended up together, and whether that
> survives a change of perplexity. **What you may not say:** that one cluster is nearer to a
> second than to a third, that a gap is wide, that a cluster is large, or anything at all
> about the axes — which is why `ProjectionPlot` hides the ticks.

### And the failure that costs people papers

Everything above had real structure in it — ten digits genuinely are ten things. Watch what
the same tool does to data with no structure whatsoever: a thousand points, fifty dimensions,
every value an independent draw from the same normal distribution. There are no groups. There
is nothing to find.

In [ ]:
from sklearn.cluster import KMeans

noise = np.random.default_rng(0).normal(size=(1000, 50))

NOISE_PERPLEXITIES = (2, 5, 30, 100)
settings = PlotSettings(
    figsize=(11, 9),
    title="Fifty dimensions of pure noise, four perplexities",
    xlabel="",
    ylabel="",
    max_cols=2,
    subplot_titles=[f"perplexity {perplexity}" for perplexity in NOISE_PERPLEXITIES],
)
host = ProjectionPlot(settings)
fig, axes = host.create_figure(n_plots=len(NOISE_PERPLEXITIES))

scores = {}
for ax, perplexity in zip(axes, NOISE_PERPLEXITIES):
    embedding = TSNE(n_components=2, perplexity=perplexity, random_state=0,
                     init="pca").fit_transform(noise)
    host.plot_on_axes(ProjectionPlot(settings), ax, coordinates=embedding, color="#777777")
    scores[perplexity] = silhouette_score(
        embedding, KMeans(5, n_init=10, random_state=0).fit_predict(embedding))

In [ ]:
raw_score = silhouette_score(noise, KMeans(5, n_init=10, random_state=0).fit_predict(noise))
print(f"k-means with k=5 on the raw 50-dimensional noise: silhouette {raw_score:.3f}")
for perplexity, score in scores.items():
    print(f"  ...on the t-SNE picture at perplexity {perplexity:>3d}: silhouette {score:.3f}")

At perplexity 2 the picture visibly breaks into little clumps. There are no clumps. The
algorithm was asked to keep each point near its two nearest neighbours, and in fifty
dimensions of noise every point has two nearest neighbours, so it obliged.

The numbers say something worse, and it holds at *every* perplexity. Run k-means on the raw
data and the silhouette is **0.016** — correctly reporting that there is nothing there. Run
the identical k-means on the two-dimensional picture and it is around **0.31**, which in any
other context you would read as decent cluster structure.

**So never cluster the embedding.** Distances in a t-SNE plot are not the data's distances;
the projection has squeezed fifty dimensions into a bounded patch of paper, and anything
measuring distance there is measuring the paper. Cluster the data, then use the projection to
look at what you found.

## What to write down

1. **Did you scale, and why.** "All columns were pixels" is a reason. "It ran either way" is
   not.
2. **The variance your picture contains** — from the scree plot, in the caption. Two
   components of 784 is a number a reader cannot guess.
3. **The perplexity**, and one other perplexity you looked at. If the finding only exists at
   one setting, it is a finding about the setting.
4. **Which claim you are making**: that points group together, or where those groups sit
   relative to each other. The first can survive. The second cannot.

---

**Where this goes next.** 06.2 keeps the digits and asks a different question — not "what
does this look like" but "can a model tell them apart", which is the check that a shape you
saw in a projection was worth believing.